In [1]:
import json
import subprocess
import numpy as np
import pandas as pd
import perf

In [2]:
perf.system.tune()

,setting,path,before,after
0,aslr,/proc/sys/kernel/randomize_va_space,off,off
1,governor,/sys/devices/system/cpu/cpu*/cpufreq/scaling_g...,performance,performance
2,energy_performance_preference,/sys/devices/system/cpu/cpu*/cpufreq/energy_pe...,performance,performance
3,cpu_frequency,/sys/devices/system/cpu/cpu*/cpufreq/scaling_{...,4.6GHz..4.6GHz;4.7GHz..4.7GHz;3.5GHz..3.5GHz,4.6GHz..4.6GHz;4.7GHz..4.7GHz;3.5GHz..3.5GHz
4,turbo,/sys/devices/system/cpu/intel_pstate/no_turbo,on,on
5,turbo_msr,/dev/cpu/*/msr,,on
6,boost,/sys/devices/system/cpu/cpufreq/boost,,on
7,rdpmc,/sys/devices/*/rdpmc,,
8,smt,/sys/devices/system/cpu/smt/control,on,on
9,nmi_watchdog,/proc/sys/kernel/nmi_watchdog,off,off


In [3]:
ADDR = 0x4200000000
PIN = {
    "branch": "predictable",
    "dcache": "hot",
    "icache": "hot",
    "dtlb": "hot",
    "itlb": "hot",
    "code": {"align": 16},
}
RUN = {"samples": 20, "iterations": 20}


def bench(
    asm,
    event=("cycles", "instructions"),
    config=None,
    data=None,
    name=None,
    backend=None,
):
    cfg = {**PIN, **RUN, **(config or {})}
    kwargs = {"backend": backend} if backend else {}
    return perf.benchmark(
        asm=asm,
        mode=["latency"],
        event=list(event),
        config=cfg,
        data=data,
        name=name,
        **kwargs,
    )


def mca(asm, cpu="native"):
    """Documented latency/throughput from llvm-mca's instruction tables."""
    text = f".intel_syntax noprefix\n{asm}\n"
    done = subprocess.run(
        ["llvm-mca", f"-mcpu={cpu}", "-iterations=1", "-json"],
        input=text,
        text=True,
        capture_output=True,
    )
    if done.returncode:
        raise RuntimeError(f"llvm-mca failed: {done.stderr.strip()}")
    region = json.loads(done.stdout)["CodeRegions"][0]
    info = region["InstructionInfoView"]["InstructionList"][0]
    return {
        "latency": info["Latency"],
        "throughput": info["RThroughput"],
        "dispatch": region["SummaryView"]["DispatchWidth"],
    }


def cache_kb(value):
    text = str(value).lower()
    return float(text.rstrip("kmb")) * (1024 if text.endswith("mb") else 1)

In [4]:
frames = []
for n in [6, 12, 24, 48, 96]:
    asm = f"mov r13d, 1000\n.L:\n" + "\n".join(["nop"] * n) + "\ndec r13d\njnz .L"
    df = bench(
        asm,
        event=("cycles", "instructions", "duration_time"),
        name=f"nop{n}",
    ).reset_index()
    df["nops"] = n
    frames.append(df)
dispatch = pd.concat(frames, ignore_index=True)
dispatch["ipc"] = dispatch["instructions"] / dispatch["cycles"]
dispatch["ghz"] = dispatch["cycles"] / dispatch["duration_time"]
clock = dispatch.groupby("nops")[["cycles", "instructions", "ipc", "ghz"]].median()
clock_ghz = float(clock["ghz"].median())
dispatch_width = float(clock["ipc"].max())
clock.round(3)

,cycles,instructions,ipc,ghz
nops,,,,
6,1247.0,8000.0,6.415,4.048
12,2191.5,14000.0,6.388,4.312
24,4191.0,26000.0,6.204,4.574
48,10013.0,50000.0,4.994,3.794
96,19011.0,98000.0,5.155,4.595


In [5]:
UNROLL = {"unroll": {"count": 64, "probes": 1, "runs": 10}}
OPS = {
    "add": ("add rax, rdi", {"regs": {"rdi": 3}}),
    "imul": ("imul rax, rdi", {"regs": {"rdi": 3}}),
    "div32": ("div ecx", {"regs": {"eax": 100, "edx": 0, "ecx": 3}}),
    "div64": ("div rcx", {"regs": {"rax": 100, "rdx": 0, "rcx": 3}}),
    "addss": ("addss xmm0, xmm0", None),
    "mulss": ("mulss xmm0, xmm0", None),
    "divss": ("divss xmm0, xmm0", None),
    "divsd": ("divsd xmm0, xmm0", None),
    "sqrtss": ("sqrtss xmm0, xmm0", None),
    "sqrtsd": ("sqrtsd xmm0, xmm0", None),
    "paddd": ("paddd xmm0, xmm0", None),
    "vpaddd": ("vpaddd ymm0, ymm0, ymm0", None),
    "lea": ("lea rax, [rax + rdi]", {"regs": {"rdi": 3}}),
    "load": ("mov rax, [rax]", {"regs": {"rax": ADDR}, "mem": {hex(ADDR): ADDR}}),
}
rows = []
for name, (asm, data) in OPS.items():
    model = mca(asm)
    df = bench(asm, config={"backend": UNROLL}, data=data, name=name, backend="unroll")
    med = df[["cycles", "instructions"]].median()
    rows.append(
        {
            "op": name,
            "measured": med["cycles"] / med["instructions"],
            "documented": model["latency"],
            "throughput": model["throughput"],
        }
    )
latency = pd.DataFrame(rows).set_index("op")
latency.round(3)

,measured,documented,throughput
op,,,
add,1.016,1,0.200
imul,3.016,3,1.000
div32,14.969,15,3.000
div64,18.031,18,3.000
addss,2.000,3,0.500
mulss,4.016,4,0.500
divss,11.016,11,1.000
divsd,13.016,14,1.000
sqrtss,12.016,12,1.000


In [6]:
LINES = 64
order = np.random.default_rng(0).permutation(LINES).tolist()
nxt = {line: order[(index + 1) % LINES] for index, line in enumerate(order)}
lines_mem = {hex(ADDR + line * 64): ADDR + nxt[line] * 64 for line in range(LINES)}
asm = "mov rax, rdi\n" + "\n".join(["mov rax, [rax]"] * LINES)
cache = bench(
    asm,
    config={"dcache": ["hot", "warm", "cool", "cold"]},
    data={"regs": {"rdi": ADDR}, "mem": lines_mem},
    name="cache",
)
cache_summary = cache.groupby("config.dcache")[["cycles", "instructions"]].median()
cache_summary = cache_summary.reindex(["hot", "warm", "cool", "cold"])
cache_summary["cycles/load"] = cache_summary["cycles"] / LINES
cache_latency = cache_summary["cycles/load"]
bw_asm = "\n".join(
    f"vmovdqu ymm0, [rdi+{line * 64}]\nvmovdqu ymm1, [rdi+{line * 64 + 32}]"
    for line in range(LINES)
)
bandwidth = bench(
    bw_asm,
    event=("cycles",),
    config={"dcache": ["hot", "warm", "cool", "cold"]},
    data={"regs": {"rdi": ADDR}, "mem": lines_mem},
    name="bandwidth",
)
bandwidth_summary = bandwidth.groupby("config.dcache").cycles.median().to_frame("cycles")
bandwidth_summary["bytes/cycle"] = LINES * 64 / bandwidth_summary["cycles"]
bandwidth_summary["GB/s"] = bandwidth_summary["bytes/cycle"] * clock_ghz
bandwidth_summary = bandwidth_summary.reindex(["hot", "warm", "cool", "cold"])
cache_bandwidth = bandwidth_summary["bytes/cycle"]
display(cache_summary.round(3))
bandwidth_summary.round(3)

,cycles,instructions,cycles/load
config.dcache,,,
hot,317.0,64.0,4.953
warm,957.0,64.0,14.953
cool,957.0,64.0,14.953
cold,20857.5,64.0,325.898


,cycles,bytes/cycle,GB/s
config.dcache,,,
hot,52.0,78.769,339.685
warm,113.0,36.248,156.315
cool,113.5,36.088,155.627
cold,1364.5,3.002,12.945


In [7]:
def tlb_mem(n, seed=1):
    order = np.random.default_rng(seed).permutation(n).tolist()
    nxt = {page: order[(index + 1) % n] for index, page in enumerate(order)}
    offset = {page: (page * 64) % 4096 for page in range(n)}
    return {
        hex(ADDR + page * 4096 + offset[page]): ADDR
        + nxt[page] * 4096
        + offset[nxt[page]]
        for page in range(n)
    }


def tlb_asm(n):
    return f"mov rax, rdi\nmov r13d, {n}\n.L:\nmov rax, [rax]\ndec r13d\njnz .L"


TLB_PAGES = [16, 32, 64, 96, 128, 192, 256]
rows = []
for n in TLB_PAGES:
    df = bench(
        tlb_asm(n),
        event=("cycles",),
        config={"dtlb": "hot"},
        data={"regs": {"rdi": ADDR}, "mem": tlb_mem(n)},
        name=f"tlb{n}",
    )
    rows.append({"pages": n, "cycles": df["cycles"].median()})
tlb_capacity = pd.DataFrame(rows).set_index("pages")
tlb_capacity["cycles/load"] = tlb_capacity["cycles"] / tlb_capacity.index
base = tlb_capacity["cycles/load"].iloc[0]
hit = tlb_capacity["cycles/load"] > 1.5 * base
first = tlb_capacity.index[hit].min() if hit.any() else None
dtlb_capacity = (
    int(tlb_capacity.index[tlb_capacity.index < first].max())
    if first is not None
    else None
)
tlb = bench(
    tlb_asm(64),
    config={"dtlb": ["hot", "cold"]},
    data={"regs": {"rdi": ADDR}, "mem": tlb_mem(64)},
    name="tlb",
)
tlb_summary = tlb.groupby("config.dtlb")[["cycles", "instructions"]].median()
tlb_summary["cycles/load"] = tlb_summary["cycles"] / 64
tlb_latency = tlb_summary["cycles/load"]
display(tlb_capacity.round(3))
tlb_summary.round(3)

,cycles,cycles/load
pages,,
16,70.0,4.375
32,151.0,4.719
64,316.5,4.945
96,493.0,5.135
128,1102.5,8.613
192,2114.0,11.010
256,2963.5,11.576


,cycles,instructions,cycles/load
config.dtlb,,,
cold,1540.0,193.0,24.062
hot,323.5,193.0,5.055


In [8]:
LOAD_REGS = 64
load_mem = {hex(ADDR + line * 64): line for line in range(LOAD_REGS)}
rows = []
for n in [1, 2, 4, 8, 16, 32, 64]:
    asm = "\n".join(f"mov rax, [rdi+{i * 64}]" for i in range(n))
    df = bench(
        asm,
        event=("cycles",),
        config={"dcache": "cold"},
        data={"regs": {"rdi": ADDR}, "mem": load_mem},
        name=f"mlp{n}",
    )
    rows.append({"loads": n, "cycles": df["cycles"].median()})
mlp = pd.DataFrame(rows).set_index("loads")
mlp["cycles/load"] = mlp["cycles"] / mlp.index
mlp_parallel = float(mlp.index.max() * cache_latency["cold"] / mlp["cycles"].max())
mlp.round(3)

,cycles,cycles/load
loads,,
1,369.5,369.500
2,373.5,186.750
4,452.0,113.000
8,447.0,55.875
16,464.0,29.000
32,808.0,25.250
64,1329.0,20.766


In [9]:
BITS = 8
VALUES = np.random.default_rng(2).integers(0, 2**BITS, 32, dtype=np.uint64).tolist()
asm = "\n".join(
    ["xor eax, eax"]
    + [
        line
        for i in range(BITS)
        for line in (
            f"test rdi, {1 << i}",
            f"jz .L{i}",
            f"add rax, {i + 1}",
            f".L{i}:",
        )
    ]
    + ["ret"]
)
branch = bench(
    asm,
    event=("cycles", "instructions", "branch-instructions", "branch-misses"),
    config={"branch": ["predictable", "unpredictable"]},
    data={"regs": {"rdi": VALUES}},
    name="branch",
)
branch_summary = branch.groupby("config.branch")[
    ["cycles", "instructions", "branch-instructions", "branch-misses"]
].mean()
delta = branch_summary.loc["unpredictable"] - branch_summary.loc["predictable"]
branch_penalty = float(delta["cycles"] / BITS)
branch_accuracy = float(
    100.0
    * (
        1.0
        - branch_summary.loc["predictable", "branch-misses"]
        / branch_summary.loc["predictable", "branch-instructions"]
    )
)
branch_summary.loc["penalty", ["cycles", "branch-misses"]] = delta[
    ["cycles", "branch-misses"]
]
branch_summary.loc["penalty/branch", "cycles"] = branch_penalty
branch_summary.round(3)

,cycles,instructions,branch-instructions,branch-misses
config.branch,,,,
predictable,5.450,19.805,8.005,0.885
unpredictable,98.855,20.067,8.000,3.630
penalty,93.405,NaN,NaN,2.745
penalty/branch,11.676,NaN,NaN,NaN


In [10]:
BTB_N = [1024, 4096, 4608, 8192]


def btb_asm(n):
    count = max(100, 1_000_000 // n)
    lines = [f"mov r13d, {count}", ".L:", "lea rax, [rip + 2]", "jmp .T0"]
    for i in range(n):
        lines += [f".T{i}:", "add rax, 8", "jmp rax", "nop", "nop"]
    lines += [".done:", "dec r13d", "jnz .L"]
    return "\n".join(lines)


rows = []
for n in BTB_N:
    count = max(100, 1_000_000 // n)
    df = bench(
        btb_asm(n),
        event=("cycles",),
        config={"samples": 10, "iterations": 10},
        name=f"btb{n}",
    )
    rows.append({"branches": n, "count": count, "cycles": df["cycles"].median()})
btb = pd.DataFrame(rows).set_index("branches")
btb["cycles/jump"] = btb["cycles"] / (btb.index * btb["count"])
base = btb["cycles/jump"].iloc[0]
hit = btb["cycles/jump"] > 1.4 * base
first = btb.index[hit].min() if hit.any() else None
btb_entries = (
    int(btb.index[btb.index < first].max()) if first is not None else None
)
display(btb.round(3))
pd.DataFrame({"btb_entries": [btb_entries]})

,count,cycles,cycles/jump
branches,,,
1024,976,2125634.0,2.127
4096,244,2145170.5,2.146
4608,217,3087512.5,3.088
8192,122,6232319.0,6.236


,btb_entries
0,4096


In [11]:
A = ADDR
B = ADDR + 0x10000
FILL = [0, 64, 128, 192, 256, 320, 384, 448, 512, 576, 640, 768, 1024]
rows = []
for n in FILL:
    asm = "mov rax, [rdi]\n" + "\n".join(["nop"] * n) + "\nmov rbx, [rsi]"
    df = bench(
        asm,
        event=("cycles",),
        config={"dcache": "cold"},
        data={"regs": {"rdi": A, "rsi": B}, "mem": {hex(A): 1, hex(B): 2}},
        name=f"rob{n}",
    )
    rows.append({"nops": n, "cycles": df["cycles"].median()})
rob = pd.DataFrame(rows).set_index("nops")
rob["ratio"] = rob["cycles"] / rob.loc[[0, 64], "cycles"].median()
hit = rob["ratio"] > 1.6
cross = rob.index[hit & hit.shift(-1, fill_value=False)]
rob_entries = int(cross.min()) if len(cross) else None
display(rob.round(3))
pd.DataFrame({"rob_entries": [rob_entries]})

,cycles,ratio
nops,,
0,250.0,0.937
64,283.5,1.063
128,328.0,1.230
192,302.0,1.132
256,299.5,1.123
320,344.5,1.291
384,316.0,1.185
448,317.5,1.190
512,583.5,2.187


,rob_entries
0,512


In [12]:
INT_REGS = ["rax", "rbx", "rcx", "rdx", "rsi", "r12", "r14", "r15"]
int_asm = "mov r13d, 1000\n.L:\n" + "\n".join(
    f"add {reg}, rdi" for reg in INT_REGS
) + "\ndec r13d\njnz .L"
int_df = bench(
    int_asm,
    event=("cycles",),
    data={"regs": {"rdi": 3}},
    name="int_alu",
)
alu_throughput = float(len(INT_REGS) * 1000 / int_df["cycles"].median())
fp_body = "\n".join(f"addss xmm{i}, xmm8" for i in range(8))
fp_asm = f"pxor xmm8, xmm8\nmov r13d, 1000\n.L:\n{fp_body}\ndec r13d\njnz .L"
fp_df = bench(fp_asm, event=("cycles",), name="fp_alu")
fp_throughput = float(8 * 1000 / fp_df["cycles"].median())
pd.DataFrame(
    {"throughput": [alu_throughput, fp_throughput]},
    index=["integer", "float"],
).round(3)

,throughput
integer,4.162
float,1.995


In [13]:
first = perf.system.info().iloc[0]


def cpu_clock_ghz(info):
    for path in (
        "/sys/devices/system/cpu/cpu0/cpufreq/cpuinfo_max_freq",
        "/sys/devices/system/cpu/cpu0/cpufreq/scaling_max_freq",
    ):
        try:
            with open(path) as fh:
                return float(fh.read().strip()) / 1_000_000
        except (OSError, ValueError):
            continue
    try:
        return float(info["freq"]) / 1e9
    except (TypeError, ValueError):
        return np.nan


mca_width = mca("nop")["dispatch"]
documented_clock = cpu_clock_ghz(first)
documented_l1 = cache_kb(first["L1d"])
documented_l2 = cache_kb(first["L2"])
documented_l3 = cache_kb(first["L3"]) / 1024
load = latency.loc["load"]
add = latency.loc["add"]
addss = latency.loc["addss"]


def metric(property, measured, documented=np.nan, unit=""):
    return (property, measured, documented, unit)


rows = [
    metric("Clock", clock_ghz, documented_clock, "GHz"),
    metric("Decode width", dispatch_width, mca_width, "µops/cycle"),
    metric("Reorder buffer", rob_entries, np.nan, "entries"),
    
    metric("Integer ALU throughput", alu_throughput, 1 / add["throughput"], "ops/cycle"),
    metric("Floating-point throughput", fp_throughput, 1 / addss["throughput"], "ops/cycle"),
    
    metric("Branch prediction accuracy", branch_accuracy, np.nan, "%"),
    metric("Branch misprediction penalty", branch_penalty, np.nan, "cycles"),
    metric("BTB entries", btb_entries, np.nan, "entries"),
    
    metric("TLBd capacity", dtlb_capacity, np.nan, "entries"),
    metric("TLBd miss", tlb_latency["cold"] - tlb_latency["hot"], np.nan, "cycles"),
    
    metric("L1d size", cache_kb(first["L1d"]), documented_l1, "KB"),
    metric("L1d load", cache_latency["hot"], load["documented"], "cycles"),
    metric("L1d bandwidth", cache_bandwidth["hot"], np.nan, "bytes/cycle"),
    metric("L2d size", cache_kb(first["L2"]), documented_l2, "KB"),
    metric("L2d load", cache_latency["warm"], np.nan, "cycles"),
    metric("L2d bandwidth", cache_bandwidth["warm"], np.nan, "bytes/cycle"),
    metric("L3d size", cache_kb(first["L3"]) / 1024, documented_l3, "MB"),
    metric("L3d load", cache_latency["cool"], np.nan, "cycles"),
    metric("L3d bandwidth", cache_bandwidth["cool"], np.nan, "bytes/cycle"),
    metric("DRAM load", cache_latency["cold"], np.nan, "cycles"),
    metric("DRAM bandwidth", cache_bandwidth["cold"] * clock_ghz, np.nan, "GB/s"),
    metric("DRAM level parallelism", mlp_parallel, np.nan, "loads"),
]
for op, row in latency.iterrows():
    rows.append(metric(op, row["measured"], row["documented"], "cycles"))
summary = pd.DataFrame(rows, columns=["property", "measured", "documented", "unit"])
summary[["measured", "documented"]] = (
    summary[["measured", "documented"]].astype(float).round(3)
)
identity = ["arch", "vendor", "model", "family", "stepping"]
for column in reversed(identity):
    summary.insert(0, column, first[column])
summary

,arch,vendor,model,family,stepping,property,measured,documented,unit
0,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,Clock,4.312,4.6,GHz
1,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,Decode width,6.415,6.0,µops/cycle
2,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,Reorder buffer,512.000,NaN,entries
3,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,Integer ALU throughput,4.162,5.0,ops/cycle
4,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,Floating-point throughput,1.995,2.0,ops/cycle
5,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,Branch prediction accuracy,88.944,NaN,%
6,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,Branch misprediction penalty,11.676,NaN,cycles
7,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,BTB entries,4096.000,NaN,entries
8,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,TLBd capacity,96.000,NaN,entries
9,x86_64,12th Gen Intel(R) Core(TM) i7-12650H,154,6,3,TLBd miss,19.008,NaN,cycles
